# NSynth Instrument Classifier

Classify instrument families from audio using the [NSynth dataset](https://magenta.withgoogle.com/datasets/nsynth).

**Models compared:**
1. Baseline: Random Forest on MFCC statistics
2. CNN: Convolutional neural network on mel spectrograms (from lab 5)
3. CNN + LSTM: CNN features + LSTM temporal modeling (from lab 5 + lab 8)

**11 Instrument Families:** bass, brass, flute, guitar, keyboard, mallet, organ, reed, string, synth_lead, vocal

In [ ]:
import json
import tarfile
import urllib.request
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
print(f"Using device: {device}")

## Constants

In [ ]:
SAMPLE_RATE = 16000
NOTE_DURATION = 4.0
N_MELS = 128
HOP_LENGTH = 512
BATCH_SIZE = 64
NUM_EPOCHS = 30
LEARNING_RATE = 1e-3
PATIENCE = 7

INSTRUMENT_FAMILIES = [
    "bass",
    "brass",
    "flute",
    "guitar",
    "keyboard",
    "mallet",
    "organ",
    "reed",
    "string",
    "synth_lead",
    "vocal",
]

DATA_DIR = Path("../data")
TRAIN_DIR = DATA_DIR / "nsynth-train"
VALID_DIR = DATA_DIR / "nsynth-valid"
TEST_DIR = DATA_DIR / "nsynth-test"
CACHE_DIR = DATA_DIR / "cache"

BASE_URL = "http://download.magenta.tensorflow.org/datasets/nsynth/"

## Data Download

Download and extract NSynth JSON+WAV splits. Each split is ~4GB.

In [ ]:
def download_and_extract(split: str, data_dir: Path):
    """Download and extract an NSynth split (train/valid/test)."""
    filename = f"nsynth-{split}.jsonwav.tar.gz"
    tar_path = data_dir / filename
    extract_dir = data_dir / f"nsynth-{split}"

    if extract_dir.exists() and (extract_dir / "examples.json").exists():
        print(f"{split} already extracted, skipping.")
        return

    data_dir.mkdir(parents=True, exist_ok=True)

    if not tar_path.exists():
        url = BASE_URL + filename
        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, tar_path)
    else:
        print(f"Tar file already exists: {tar_path}")

    print(f"Extracting {tar_path} ...")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=data_dir, filter="data")
    tar_path.unlink()
    print(f"Done: {extract_dir}")


def download_all():
    for split in ["train", "valid", "test"]:
        download_and_extract(split, DATA_DIR)

In [ ]:
download_all()

## Data Loading

In [ ]:
def load_nsynth_json(split_dir: Path) -> pd.DataFrame:
    """Load NSynth metadata from examples.json into a DataFrame."""
    json_path = split_dir / "examples.json"
    with open(json_path) as f:
        data = json.load(f)

    rows = []
    for note_str, meta in data.items():
        rows.append(
            {
                "note_str": note_str,
                "instrument_family": meta["instrument_family"],
                "instrument_family_str": meta["instrument_family_str"],
                "instrument_source": meta["instrument_source"],
                "instrument_source_str": meta["instrument_source_str"],
                "instrument_str": meta["instrument_str"],
                "pitch": meta["pitch"],
                "velocity": meta["velocity"],
            }
        )

    return pd.DataFrame(rows)


def load_audio(note_str: str, split_dir: Path) -> np.ndarray:
    """Load a single WAV file and return the waveform."""
    wav_path = split_dir / "audio" / f"{note_str}.wav"
    audio, _ = librosa.load(wav_path, sr=SAMPLE_RATE)
    return audio

In [ ]:
df_train = load_nsynth_json(TRAIN_DIR)
df_valid = load_nsynth_json(VALID_DIR)
df_test = load_nsynth_json(TEST_DIR)
print(f"Train: {len(df_train):,} | Valid: {len(df_valid):,} | Test: {len(df_test):,}")

## Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (df, title) in zip(
    axes, [(df_train, "Train"), (df_valid, "Valid"), (df_test, "Test")], strict=True
):
    counts = df["instrument_family_str"].value_counts().reindex(INSTRUMENT_FAMILIES)
    counts.plot.bar(ax=ax, color="steelblue")
    ax.set_title(f"{title} ({len(df):,} notes)")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, family in enumerate(INSTRUMENT_FAMILIES[:10]):
    sample = df_train[df_train["instrument_family_str"] == family].iloc[0]
    audio = load_audio(sample["note_str"], TRAIN_DIR)
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_mels=N_MELS,
        hop_length=HOP_LENGTH,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(
        mel_db,
        sr=SAMPLE_RATE,
        hop_length=HOP_LENGTH,
        x_axis="time",
        y_axis="mel",
        ax=axes[i],
        cmap="magma",
    )
    axes[i].set_title(f"{family} (pitch={sample['pitch']})")

axes[-1].axis("off")
plt.tight_layout()
plt.show()

## Feature Extraction

Extract log-mel spectrograms and cache them to disk for fast loading during training.

In [ ]:
def extract_mel_spectrogram(
    audio: np.ndarray, sr: int = SAMPLE_RATE, n_mels: int = N_MELS, hop_length: int = HOP_LENGTH
) -> np.ndarray:
    """Extract log-mel spectrogram from audio waveform."""
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels, hop_length=hop_length)
    return librosa.power_to_db(mel, ref=np.max)

In [ ]:
def cache_mel_spectrograms(df: pd.DataFrame, split_dir: Path, cache_name: str) -> np.ndarray:
    """Precompute and cache mel spectrograms as .npy files."""
    cache_path = CACHE_DIR / cache_name
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    if cache_path.exists():
        print(f"Loading cached spectrograms from {cache_path}")
        return np.load(cache_path)

    specs = []
    for i, row in df.iterrows():
        if i % 5000 == 0:
            print(f"  Processing {i}/{len(df)}...")
        audio = load_audio(row["note_str"], split_dir)
        mel = extract_mel_spectrogram(audio)
        specs.append(mel)

    specs = np.array(specs, dtype=np.float32)
    np.save(cache_path, specs)
    print(f"Cached {specs.shape} to {cache_path}")
    return specs

In [ ]:
print("Extracting train spectrograms...")
X_train = cache_mel_spectrograms(df_train, TRAIN_DIR, "mel_train.npy")
print("Extracting valid spectrograms...")
X_valid = cache_mel_spectrograms(df_valid, VALID_DIR, "mel_valid.npy")
print("Extracting test spectrograms...")
X_test = cache_mel_spectrograms(df_test, TEST_DIR, "mel_test.npy")

le = LabelEncoder()
y_train = le.fit_transform(df_train["instrument_family_str"])
y_valid = le.transform(df_valid["instrument_family_str"])
y_test = le.transform(df_test["instrument_family_str"])

print(f"\nX_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_valid: {X_valid.shape}, y_valid: {y_valid.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")
print(f"Classes: {le.classes_}")

## PyTorch Dataset & DataLoader

In [ ]:
class NSynthDataset(Dataset):
    def __init__(self, spectrograms: np.ndarray, labels: np.ndarray):
        self.spectrograms = torch.from_numpy(spectrograms).unsqueeze(1)
        self.labels = torch.from_numpy(labels).long()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.spectrograms[idx], self.labels[idx]

In [ ]:
train_dataset = NSynthDataset(X_train, y_train)
valid_dataset = NSynthDataset(X_valid, y_valid)
test_dataset = NSynthDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

sample_x, sample_y = next(iter(train_loader))
print(f"Batch shape: {sample_x.shape}, Labels shape: {sample_y.shape}")

## Model 1: Baseline (Random Forest on MFCCs)

Quick baseline using handcrafted features. No GPU needed.

In [ ]:
def extract_mfcc_stats(audio: np.ndarray, sr: int = SAMPLE_RATE, n_mfcc: int = 20) -> np.ndarray:
    """Extract MFCC statistics (mean and std per coefficient)."""
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])

In [ ]:
rf_accuracy = 0.0
print("Extracting MFCC features for baseline (this may take a while)...")

MAX_BASELINE_SAMPLES = 10000
train_subset = df_train.iloc[:MAX_BASELINE_SAMPLES]
X_train_mfcc = np.array(
    [
        extract_mfcc_stats(load_audio(row["note_str"], TRAIN_DIR))
        for _, row in train_subset.iterrows()
    ]
)
y_train_subset = y_train[:MAX_BASELINE_SAMPLES]

print(f"Training RF on {X_train_mfcc.shape[0]} samples...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_mfcc, y_train_subset)

val_subset = df_valid.iloc[:2000]
X_val_mfcc = np.array(
    [extract_mfcc_stats(load_audio(row["note_str"], VALID_DIR)) for _, row in val_subset.iterrows()]
)
y_val_subset = y_valid[:2000]

rf_accuracy = rf_model.score(X_val_mfcc, y_val_subset)
print(f"Baseline RF accuracy: {rf_accuracy:.2%}")

## Model 2: CNN (from lab 5)

Convolutional neural network on mel spectrograms.

In [ ]:
class InstrumentCNN(nn.Module):
    def __init__(self, num_classes: int = 11):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


cnn_model = InstrumentCNN(num_classes=len(INSTRUMENT_FAMILIES)).to(device)
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f"CNN parameters: {total_params:,}")
print(cnn_model)

## Model 3: CNN + LSTM (from lab 5 + lab 8)

CNN extracts spatial features from spectrogram frames, LSTM models temporal dependencies.

In [ ]:
class CNNLSTM(nn.Module):
    def __init__(self, num_classes: int = 11, input_shape=(1, 128, 126)):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )
        with torch.no_grad():
            dummy = self.cnn(torch.zeros(1, *input_shape))
            self._seq_len = dummy.size(2)
            lstm_input_size = dummy.size(1) * dummy.size(3)
        self.lstm = nn.LSTM(
            input_size=lstm_input_size,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.cnn(x)
        batch_size = x.size(0)
        x = x.permute(0, 2, 1, 3).reshape(batch_size, self._seq_len, -1)
        _, (hidden, _) = self.lstm(x)
        x = hidden[-1]
        return self.classifier(x)


cnnlstm_model = CNNLSTM(num_classes=len(INSTRUMENT_FAMILIES)).to(device)
total_params = sum(p.numel() for p in cnnlstm_model.parameters())
print(f"CNN+LSTM parameters: {total_params:,}")
print(cnnlstm_model)

## Training

In [ ]:
def train_model(model, train_loader, valid_loader, num_epochs=NUM_EPOCHS, patience=PATIENCE):
    """Train a model with validation and early stopping."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=3,
    )

    best_val_acc = 0.0
    best_model_state = None
    patience_counter = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_loss /= val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        print(
            f"Epoch {epoch + 1:2d}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.6f}"
            f"{' *' if val_acc == best_val_acc else ''}"
        )

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    print(f"\nBest validation accuracy: {best_val_acc:.4f}")
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model, history

In [ ]:
print("Training CNN...")
cnn_model = InstrumentCNN(num_classes=len(INSTRUMENT_FAMILIES))
cnn_model, cnn_history = train_model(cnn_model, train_loader, valid_loader)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(cnn_history["train_loss"], label="Train")
axes[0].plot(cnn_history["val_loss"], label="Validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("CNN Loss")
axes[0].legend()

axes[1].plot(cnn_history["train_acc"], label="Train")
axes[1].plot(cnn_history["val_acc"], label="Validation")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("CNN Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print("Training CNN+LSTM...")
cnnlstm_model = CNNLSTM(num_classes=len(INSTRUMENT_FAMILIES))
cnnlstm_model, cnnlstm_history = train_model(cnnlstm_model, train_loader, valid_loader)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(cnnlstm_history["train_loss"], label="Train")
axes[0].plot(cnnlstm_history["val_loss"], label="Validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("CNN+LSTM Loss")
axes[0].legend()

axes[1].plot(cnnlstm_history["train_acc"], label="Train")
axes[1].plot(cnnlstm_history["val_acc"], label="Validation")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("CNN+LSTM Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

## Evaluation on Test Set

In [ ]:
@torch.no_grad()
def evaluate_model(model, test_loader):
    """Evaluate model on test set and return predictions and labels."""
    model.eval()
    all_preds = []
    all_labels = []

    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)

In [ ]:
print("Evaluating CNN...")
cnn_preds, cnn_labels = evaluate_model(cnn_model, test_loader)
print(classification_report(cnn_labels, cnn_preds, target_names=le.classes_))

print("\n" + "=" * 60 + "\n")

print("Evaluating CNN+LSTM...")
cnnlstm_preds, cnnlstm_labels = evaluate_model(cnnlstm_model, test_loader)
print(classification_report(cnnlstm_labels, cnnlstm_preds, target_names=le.classes_))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

cm_cnn = confusion_matrix(cnn_labels, cnn_preds)
disp_cnn = ConfusionMatrixDisplay(confusion_matrix=cm_cnn, display_labels=le.classes_)
disp_cnn.plot(ax=axes[0], cmap="Blues", xticks_rotation=45)
axes[0].set_title("CNN Confusion Matrix")

cm_cnnlstm = confusion_matrix(cnnlstm_labels, cnnlstm_preds)
disp_cnnlstm = ConfusionMatrixDisplay(confusion_matrix=cm_cnnlstm, display_labels=le.classes_)
disp_cnnlstm.plot(ax=axes[1], cmap="Greens", xticks_rotation=45)
axes[1].set_title("CNN+LSTM Confusion Matrix")

plt.tight_layout()
plt.show()

In [ ]:
results = {
    "Baseline (RF)": rf_accuracy,
    "CNN": (cnn_preds == cnn_labels).mean(),
    "CNN+LSTM": (cnnlstm_preds == cnnlstm_labels).mean(),
}

fig, ax = plt.subplots(figsize=(10, 5))
pd.Series(results).plot.bar(ax=ax, color=["lightblue", "steelblue", "darkblue"])
ax.set_ylabel("Test Accuracy")
ax.set_title("Model Comparison")
ax.set_ylim(0, 1)
for i, v in enumerate(results.values()):
    ax.text(i, v + 0.02, f"{v:.2%}", ha="center")
plt.tight_layout()
plt.show()

## Inference: Classify a Given Sound

In [ ]:
@torch.no_grad()
def predict_instrument(
    audio_path: str | Path,
    model=None,
    top_k: int = 5,
    audio: np.ndarray | None = None,
) -> list[dict]:
    """Predict instrument family for a given audio file."""
    if model is None:
        model = cnnlstm_model
    model.eval()
    if audio is None:
        audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE)

    if len(audio) > SAMPLE_RATE * NOTE_DURATION:
        audio = audio[: int(SAMPLE_RATE * NOTE_DURATION)]
    elif len(audio) < SAMPLE_RATE * NOTE_DURATION:
        audio = np.pad(audio, (0, int(SAMPLE_RATE * NOTE_DURATION) - len(audio)))

    mel = extract_mel_spectrogram(audio)
    mel_tensor = torch.from_numpy(mel).float().unsqueeze(0).unsqueeze(0).to(device)

    logits = model(mel_tensor)
    probs = torch.softmax(logits, dim=1).squeeze().cpu().numpy()

    top_indices = np.argsort(probs)[::-1][:top_k]
    predictions = []
    for idx in top_indices:
        predictions.append(
            {
                "family": le.classes_[idx],
                "confidence": float(probs[idx]),
            }
        )
    return predictions

In [ ]:
def visualize_prediction(audio_path: str | Path, top_k: int = 5):
    """Predict and visualize instrument classification for an audio file."""
    audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE)
    predictions = predict_instrument(audio_path, top_k=top_k, audio=audio)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_mels=N_MELS,
        hop_length=HOP_LENGTH,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(
        mel_db,
        sr=SAMPLE_RATE,
        hop_length=HOP_LENGTH,
        x_axis="time",
        y_axis="mel",
        ax=axes[0],
        cmap="magma",
    )
    axes[0].set_title(f"Input: {Path(audio_path).name}")

    families = [p["family"] for p in predictions]
    confidences = [p["confidence"] for p in predictions]
    colors = ["steelblue" if i > 0 else "coral" for i in range(len(families))]
    axes[1].barh(families[::-1], confidences[::-1], color=colors[::-1])
    axes[1].set_xlabel("Confidence")
    top = predictions[0]
    axes[1].set_title(f"Prediction: {top['family']} ({top['confidence']:.1%})")
    axes[1].set_xlim(0, 1)

    plt.tight_layout()
    plt.show()

    print("\nTop predictions:")
    for p in predictions:
        print(f"  {p['family']:12s} {p['confidence']:.1%}")

    return predictions

In [ ]:
import ipywidgets as widgets
from IPython.display import Audio, display

file_input = widgets.Text(
    value=str(TEST_DIR / "audio" / f"{df_test.iloc[0]['note_str']}.wav"),
    description="Audio path:",
    layout=widgets.Layout(width="80%"),
)
classify_btn = widgets.Button(description="Classify", button_style="primary")
output = widgets.Output()


def on_classify(btn):
    output.clear_output()
    with output:
        path = Path(file_input.value)
        if not path.exists():
            print(f"File not found: {path}")
            return
        display(Audio(filename=str(path)))
        visualize_prediction(path)


classify_btn.on_click(on_classify)
display(widgets.VBox([file_input, classify_btn, output]))